# 02 — Chunking Experiment

Sweep chunk_size and chunk_overlap to inspect chunk count and per-chunk length distribution. Verify metadata is preserved across configurations.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from app.services.pdf_loader import load_pdf_pages
from app.services.text_chunker import chunk_pages

In [ ]:
PDF_DIR = Path('../data/raw/uploaded_pdfs')
pdfs = sorted(PDF_DIR.glob('*.pdf'))
all_pages = []
for p in pdfs:
    all_pages.extend(load_pdf_pages(p))
print(f'Loaded {len(all_pages)} pages across {len(pdfs)} PDFs')

In [ ]:
configs = [
    (400, 50),
    (600, 100),
    (800, 150),
    (1200, 200),
]
rows = []
for size, overlap in configs:
    chunks = chunk_pages(all_pages, size, overlap)
    lens = [len(c.text) for c in chunks]
    rows.append({
        'chunk_size': size,
        'overlap': overlap,
        'n_chunks': len(chunks),
        'mean_len': round(sum(lens)/len(lens), 1) if lens else 0,
        'max_len': max(lens) if lens else 0,
        'min_len': min(lens) if lens else 0,
    })
df = pd.DataFrame(rows)
df

In [ ]:
# Metadata sanity check at the default settings
chunks = chunk_pages(all_pages, 800, 150)
for c in chunks[:3]:
    md = c.metadata
    print(md['file_name'], 'p', md['page_number'], '|', md['chunk_id'], '|', len(c.text), 'chars')
    assert md['file_name'] and md['page_number'] and md['chunk_id']

**Conclusion**: `chunk_size=800`, `overlap=150` produces a reasonable distribution (~41 chunks for the 5 demo PDFs). Smaller chunks inflate the index without improving retrieval on lecture-style content; larger chunks dilute LLM context.